<a href="https://colab.research.google.com/github/Ananyaya17/Hindi-Image-Captioning-/blob/main/beam_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1 — Environment Setup (FINAL)
# ============================================================

# Verify GPU
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

print("GPU Available:", tf.config.list_physical_devices('GPU'))


# Enable mixed precision (safe performance boost)
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

print("Mixed precision enabled")


# Install required packages
!pip install -q nltk tqdm


# Imports
import os

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import json

import pickle

from tqdm import tqdm


import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.utils import to_categorical


print("\n✅ Environment ready")

TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision enabled

✅ Environment ready


In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

print("\nChecking Drive access...")

print("Drive exists:",os.path.exists("/content/drive/MyDrive"))

print("\nSample files:")

print(os.listdir("/content/drive/MyDrive")[:10])

print("\n✅ Drive mounted successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Checking Drive access...
Drive exists: True

Sample files:
['syllabus first year.pdf', 'syllabus first year.gdoc', 'IGDTUW MUN report-1.docx', 'Classroom', 'Documents', 'Prog in C file', 'Chem assignment.pdf', 'ieee membership card.PDF', 'Screenshot_2023-04-01-22-40-14-18_1c337646f29875672b5a61192b9010f9.jpg', 'Screenshot_2023-04-01-22-41-03-73_1c337646f29875672b5a61192b9010f9.jpg']

✅ Drive mounted successfully


In [ ]:
# ============================================================
# CELL 3 — Extract Dataset
# ============================================================

import os
import zipfile
import shutil

# Dataset zip location (same as your previous work)
zip_path = '/content/drive/MyDrive/data.zip'

extract_path = '/content/dataset'


# Create dataset folder
if not os.path.exists(extract_path):

    os.makedirs(extract_path)


# Check dataset exists
if not os.path.exists(zip_path):

    raise FileNotFoundError("❌ data.zip not found")


print("Extracting dataset...")


with zipfile.ZipFile(zip_path,'r') as zip_ref:

    zip_ref.extractall(extract_path)


print("✅ Extraction complete")


# Handle archive folder automatically
archive_path = os.path.join(extract_path,'archive (1)')


if os.path.exists(archive_path):

    images_src = os.path.join(
        archive_path,
        'Images_all'
    )

    captions_src = os.path.join(
        archive_path,
        'Clean-5Sentences_withComma.txt'
    )


    images_dst = os.path.join(
        extract_path,
        'Images_all'
    )

    captions_dst = os.path.join(
        extract_path,
        'captions.txt'
    )


    if os.path.exists(images_src):

        if os.path.exists(images_dst):

            shutil.rmtree(images_dst)

        shutil.move(images_src,images_dst)

        print("✅ Images moved")


    if os.path.exists(captions_src):

        if os.path.exists(captions_dst):

            os.remove(captions_dst)

        shutil.move(captions_src,captions_dst)

        print("✅ Captions moved")


    shutil.rmtree(archive_path)



print("\nVerifying dataset")


image_folder = os.path.join(
    extract_path,
    'Images_all'
)

caption_file = os.path.join(
    extract_path,
    'captions.txt'
)


print("Images folder exists:",os.path.exists(image_folder))

print("Captions file exists:",os.path.exists(caption_file))


if os.path.exists(image_folder):

    images = os.listdir(image_folder)

    print("Total images:",len(images))

    print("Sample images:",images[:5])


if os.path.exists(caption_file):

    with open(caption_file,'r',encoding='utf-8') as f:

        lines = f.readlines()

    print("Total captions:",len(lines))

    print("Sample caption:",lines[1][:120])


print("\n✅ Dataset ready")

Extracting dataset...
✅ Extraction complete
✅ Images moved
✅ Captions moved

Verifying dataset
Images folder exists: True
Captions file exists: True
Total images: 8091
Sample images: ['3533394378_1513ec90db.jpg', '442594271_2c3dd38483.jpg', '2045023435_181854c013.jpg', '3465791729_5bf9bd8635.jpg', '2920969723_83918fe909.jpg']
Total captions: 40451
Sample caption: 1000268201_693b08cb0e,गुलाबी पोशाक में बच्चा प्रवेश के रास्ते में सीढ़ियों के सेट पर चढ़ रहा है।


✅ Dataset ready


In [ ]:
# ============================================================
# FIXED CELL 4 — Clean Captions + Train/Val/Test Split
# Fix: Unicode normalization for consistent Hindi vocabulary
# ============================================================

import os
import random
import json
import re
import unicodedata

dataset_path = "/content/dataset"
image_path   = os.path.join(dataset_path, "Images_all")
caption_path = os.path.join(dataset_path, "captions.txt")


def clean_hindi_caption(caption):
    """
    Normalize Unicode and strip punctuation while
    preserving all Hindi characters and spaces.
    Without this, the same Hindi word can appear as
    multiple vocabulary entries due to Unicode variants.
    """
    # NFC normalization — critical for Hindi consistency
    caption = unicodedata.normalize('NFC', caption)
    # Remove punctuation but keep Devanagari and spaces
    caption = re.sub(r'[।,.!?"\'\-]', '', caption)
    # Collapse multiple spaces
    caption = re.sub(r'\s+', ' ', caption).strip()
    return caption


captions = {}

with open(caption_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if ',' not in line:
            continue
        img, caption = line.split(',', 1)
        img     = img.strip() + '.jpg'
        caption = clean_hindi_caption(caption.strip())
        caption = 'startseq ' + caption + ' endseq'

        if img not in captions:
            captions[img] = []
        captions[img].append(caption)

print("Total captioned images:", len(captions))

# Remove images missing from disk
valid_captions = {
    img: caps
    for img, caps in captions.items()
    if os.path.exists(os.path.join(image_path, img))
}

print("Valid images:", len(valid_captions))

# Create image list and shuffle
image_list = list(valid_captions.keys())
random.seed(42)
random.shuffle(image_list)

# Split dataset
train_size = int(len(image_list) * 0.70)
val_size   = int(len(image_list) * 0.15)

train_images = image_list[:train_size]
val_images   = image_list[train_size:train_size + val_size]
test_images  = image_list[train_size + val_size:]

print("\nDataset split:")
print("Train:", len(train_images))
print("Validation:", len(val_images))
print("Test:", len(test_images))

# Save splits and captions
splits = {"train": train_images, "val": val_images, "test": test_images}

with open("/content/dataset_splits.json", "w") as f:
    json.dump(splits, f)

with open("/content/clean_captions.json", "w", encoding='utf-8') as f:
    json.dump(valid_captions, f, ensure_ascii=False)

print("\n✅ Splits and captions saved")

# Show vocabulary size improvement
all_words = set()
for caps in valid_captions.values():
    for cap in caps:
        all_words.update(cap.split())
print(f"Unique tokens after normalization: {len(all_words)}")

sample_img = train_images[0]
print("\nSample image:", sample_img)
print("Sample captions:")
for cap in valid_captions[sample_img][:2]:
    print(" ", cap)


Total captioned images: 8091
Valid images: 8090

Dataset split:
Train: 5663
Validation: 1213
Test: 1214

✅ Splits and captions saved
Unique tokens after normalization: 9584

Sample image: 2875528143_94d9480fdd.jpg
Sample captions:
  startseq गुलाबी रंग की लड़की बिस्तर पर उछलती है endseq
  startseq गुलाबी लिओटार्ड में छोटी लड़की कूद रही है और नीले बिस्तर पर स्ट्रैडल विभाजन में उतर रही है endseq


In [ ]:
# ============================================================
# FIXED CELL 5 — Tokenizer + Vocabulary Setup
# Fix: use sparse integer labels instead of to_categorical
#      this removes the need for one-hot vectors entirely
# ============================================================

import json
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load data
with open("/content/clean_captions.json", "r", encoding='utf-8') as f:
    captions = json.load(f)

with open("/content/dataset_splits.json", "r") as f:
    splits = json.load(f)

train_images = splits["train"]
val_images   = splits["val"]
test_images  = splits["test"]

# Collect TRAIN captions only (never fit tokenizer on val/test)
train_captions = [
    cap
    for img in train_images
    for cap in captions[img]
]

print("Total train captions:", len(train_captions))

# Tokenizer — lower=False to preserve Hindi casing
# filters removes only symbols, not Hindi characters
tokenizer = Tokenizer(
    oov_token="<unk>",
    filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~\t\n',
    lower=False
)
tokenizer.fit_on_texts(train_captions)

# Remove very rare words (appear < 2 times) — reduces vocab noise
word_counts = tokenizer.word_counts
filtered_words = {
    w for w, c in word_counts.items() if c >= 2
}

# Refit with only frequent words
tokenizer = Tokenizer(
    oov_token="<unk>",
    filters='!"#$%&()*+.,-/:;=?@[\\]^_`{|}~\t\n',
    lower=False
)
tokenizer.fit_on_texts([
    ' '.join(w for w in cap.split() if w in filtered_words or w in ('startseq', 'endseq'))
    for cap in train_captions
])

vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size (after rare word removal):", vocab_size)

# Max caption length
max_length = max(len(c.split()) for c in train_captions)
print("Max caption length:", max_length)

# Save tokenizer and config
with open("/content/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

config = {"vocab_size": vocab_size, "max_length": max_length}
with open("/content/model_config.json", "w") as f:
    json.dump(config, f)

print("\n✅ Tokenizer saved")
print("✅ Config saved")

# Example
example = train_captions[0]
seq = tokenizer.texts_to_sequences([example])[0]
print("\nExample caption:", example)
print("Tokenized (first 15):", seq[:15])


Total train captions: 28315
Vocabulary size (after rare word removal): 4668
Max caption length: 37

✅ Tokenizer saved
✅ Config saved

Example caption: startseq गुलाबी रंग की लड़की बिस्तर पर उछलती है endseq
Tokenized (first 15): [2, 95, 18, 10, 23, 361, 7, 1332, 4, 3]


In [ ]:
# ============================================================
# CELL 6 — Spatial Feature Extraction (FINAL OPTIMIZED)
# ============================================================

import tensorflow as tf

import numpy as np

import os

from tqdm import tqdm

from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

from tensorflow.keras.preprocessing import image


IMAGE_PATH="/content/dataset/Images_all"


print("Loading InceptionV3...")


base_model=InceptionV3(

    weights='imagenet',

    include_top=False
)


print("CNN loaded")


print("Output shape:",base_model.output.shape)


def extract_feature(img_path):

    img=image.load_img(

        img_path,

        target_size=(299,299)
    )

    img=image.img_to_array(img)

    img=np.expand_dims(img,axis=0)

    img=preprocess_input(img)


    feature=base_model.predict(

        img,

        verbose=0
    )


    feature=np.reshape(

        feature,

        (64,2048)
    )


    return feature.astype("float32")



features={}

images=os.listdir(IMAGE_PATH)


print("\nExtracting features...")


for img in tqdm(images):

    try:

        path=os.path.join(

            IMAGE_PATH,

            img
        )

        features[img]=extract_feature(path)

    except:

        print("Skipped:",img)



print("\nFeature extraction done")


print("Total features:",len(features))


# Save
np.save(

    "/content/spatial_features.npy",

    features
)


print("✅ Features saved")

Loading InceptionV3...
CNN loaded
Output shape: (None, None, None, 2048)

Extracting features...


100%|██████████| 8091/8091 [12:47<00:00, 10.54it/s]



Feature extraction done
Total features: 8091
✅ Features saved


In [ ]:
# ============================================================
# FIXED CELL 7 — Training Data Generator
# Fix: yield integer labels (not one-hot) — works with
#      sparse_categorical_crossentropy which is much faster
#      and uses far less memory
# ============================================================

import numpy as np
import random
from tensorflow.keras.preprocessing.sequence import pad_sequences


def data_generator(image_list, batch_size=64):
    """
    Yields batches of (image_features, partial_caption) → next_word_index.
    Uses integer labels (not one-hot) for sparse_categorical_crossentropy.
    Shuffles images each epoch for better generalization.
    """
    X1, X2, y = [], [], []
    n = 0

    while True:
        # Shuffle each epoch
        imgs = image_list.copy()
        random.shuffle(imgs)

        for img in imgs:
            if img not in features:
                continue

            feature = features[img]

            for caption in captions[img]:
                seq = tokenizer.texts_to_sequences([caption])[0]

                for i in range(1, len(seq)):
                    in_seq  = pad_sequences([seq[:i]], maxlen=max_length)[0]
                    out_seq = seq[i]   # integer label, NOT one-hot

                    X1.append(feature.astype('float32'))
                    X2.append(in_seq.astype('float32'))
                    y.append(out_seq)   # single integer

                    n += 1

                    if n == batch_size:
                        yield (
                            (
                                np.array(X1, dtype='float32'),
                                np.array(X2, dtype='float32')
                            ),
                            np.array(y, dtype='int32')   # integer array
                        )
                        X1, X2, y, n = [], [], [], 0


print("✅ Generator ready (sparse integer labels)")

# Test one batch
gen    = data_generator(train_images, 4)
sample = next(gen)

print("\nSample batch shapes:")
print("Image features:", sample[0][0].shape)   # (4, 64, 2048)
print("Caption input: ", sample[0][1].shape)   # (4, max_length)
print("Labels:        ", sample[1].shape)       # (4,) — integers not one-hot
print("Label dtype:   ", sample[1].dtype)
print("Sample labels: ", sample[1])             # e.g. [42, 17, 305, 8]


✅ Generator ready (sparse integer labels)

Sample batch shapes:
Image features: (4, 64, 2048)
Caption input:  (4, 37)
Labels:         (4,)
Label dtype:    int32
Sample labels:  [88 85 37  5]


In [ ]:
# ============================================================
# FIXED CELL 8 — Model (same architecture, fixed compile)
# Fix: compile with sparse_categorical_crossentropy
#      to match the integer labels from the generator
# ============================================================

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, LSTM, Embedding, Dropout,
    LayerNormalization, Add, MultiHeadAttention,
    GlobalAveragePooling1D, Bidirectional, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


class BahdanauAttention(tf.keras.layers.Layer):

    def __init__(self, units, **kwargs):
        super(BahdanauAttention, self).__init__(**kwargs)
        self.units = units
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V  = Dense(1)

    def call(self, features, hidden):
        hidden_expanded    = tf.expand_dims(hidden, 1)
        score              = tf.nn.tanh(self.W1(features) + self.W2(hidden_expanded))
        attention_weights  = tf.nn.softmax(self.V(score), axis=1)
        context_vector     = tf.reduce_sum(attention_weights * features, axis=1)
        return context_vector

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


def build_final_model(vocab_size, max_length):

    # ── IMAGE ENCODER ──────────────────────────────────────
    image_input = Input(shape=(64, 2048))

    img = Dense(512, activation='relu')(image_input)

    # Image self-attention
    img_att = MultiHeadAttention(num_heads=8, key_dim=64)(img, img)
    img     = Add()([img, img_att])
    img     = LayerNormalization()(img)

    # Transformer-style FFN
    img_ff  = Dense(512, activation='relu')(img)
    img_ff  = Dense(512)(img_ff)
    img     = Add()([img, img_ff])
    img     = LayerNormalization()(img)

    image_context = GlobalAveragePooling1D()(img)

    # ── TEXT ENCODER ───────────────────────────────────────
    text_input = Input(shape=(max_length,))

    text = Embedding(vocab_size, 256, mask_zero=False)(text_input)
    text = Bidirectional(LSTM(256, dropout=0.3, return_sequences=False))(text)
    text = Dense(512, activation='relu')(text)

    # ── CROSS ATTENTION ────────────────────────────────────
    attention      = BahdanauAttention(512)
    context_vector = attention(img, text)

    # ── MULTIMODAL FUSION ──────────────────────────────────
    merged = Concatenate()([context_vector, text, image_context])
    merged = Dense(1024, activation='relu')(merged)
    merged = Dropout(0.4)(merged)
    merged = Dense(512, activation='relu')(merged)
    merged = LayerNormalization()(merged)

    # ── OUTPUT ─────────────────────────────────────────────
    # dtype='float32' required for mixed precision stability
    outputs = Dense(vocab_size, activation='softmax', dtype='float32')(merged)

    model = Model(inputs=[image_input, text_input], outputs=outputs)
    return model


model = build_final_model(vocab_size, max_length)

# KEY FIX: sparse_categorical_crossentropy matches integer labels
# from the generator — no to_categorical needed anywhere
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=Adam(learning_rate=2e-4, clipnorm=1.0),
    metrics=['accuracy']
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 64, 2048)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 64, 512)   │  1,049,088 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 64, 512)   │  1,050,624 │ dense_13[0][0],   │
│ (MultiHeadAttentio… │                   │            │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 64, 512)   │          0 │ dense_13[0][0],   │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 64, 512)   │      1,024 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 64, 512)   │    262,656 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 37)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 64, 512)   │    262,656 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 37, 256)   │  1,195,008 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 64, 512)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_15[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 512)       │  1,050,624 │ embedding_1[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 64, 512)   │      1,024 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 512)       │    262,656 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bahdanau_attention… │ (None, 512)       │    525,825 │ layer_normalizat… │
│ (BahdanauAttention) │                   │            │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 1536)      │          0 │ bahdanau_attenti… │
│ (Concatenate)       │                   │            │ dense_16[0][0],   │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 1024)      │  1,573,888 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 1024)      │          0 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 10,155,581 (38.74 MB)

 Trainable params: 10,155,581 (38.74 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ============================================================
# FIXED CELL 9 — Training Pipeline
# Fix: update TF dataset output_signature to match
#      integer labels (int32 shape=(None,) not float32 one-hot)
# ============================================================

import tensorflow as tf

BATCH_SIZE = 64

train_gen = data_generator(train_images, BATCH_SIZE)
val_gen   = data_generator(val_images,   BATCH_SIZE)


def count_samples(image_list):
    total = 0
    for img in image_list:
        if img not in captions:
            continue
        for cap in captions[img]:
            seq    = tokenizer.texts_to_sequences([cap])[0]
            total += (len(seq) - 1)
    return total


print("Counting samples...")
train_samples = count_samples(train_images)
val_samples   = count_samples(val_images)
test_samples  = count_samples(test_images)

train_steps = train_samples // BATCH_SIZE
val_steps   = val_samples   // BATCH_SIZE
test_steps  = test_samples  // BATCH_SIZE

print("Train samples:", train_samples, "| steps:", train_steps)
print("Val samples:  ", val_samples,   "| steps:", val_steps)
print("Test samples: ", test_samples,  "| steps:", test_steps)


# FIXED output_signature — labels are int32 shape (None,)
# not float32 shape (None, vocab_size)
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_images, BATCH_SIZE),
    output_signature=(
        (
            tf.TensorSpec(shape=(None, 64, 2048),    dtype=tf.float32),
            tf.TensorSpec(shape=(None, max_length),   dtype=tf.float32)
        ),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)   # integer labels
    )
).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(val_images, BATCH_SIZE),
    output_signature=(
        (
            tf.TensorSpec(shape=(None, 64, 2048),    dtype=tf.float32),
            tf.TensorSpec(shape=(None, max_length),   dtype=tf.float32)
        ),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)   # integer labels
    )
).prefetch(tf.data.AUTOTUNE)

print("\n✅ Training pipeline ready")



Counting samples...
Train samples: 327582 | steps: 5118
Val samples:   70100 | steps: 1095
Test samples:  70569 | steps: 1102

✅ Training pipeline ready


In [ ]:
# ============================================================
# CELL 10 — FINAL TRAINING SETUP
# ============================================================

from tensorflow.keras.callbacks import (

    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)


EPOCHS=30



checkpoint=ModelCheckpoint(

    "/content/hybrid_best.keras",

    monitor="val_accuracy",

    save_best_only=True,

    mode="max",

    verbose=1
)



early_stop=EarlyStopping(

    monitor="val_loss",

    patience=6,

    restore_best_weights=True
)



reduce_lr=ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=3,

    verbose=1,

    min_lr=1e-6
)



print("Callbacks ready")

print("Train steps:",train_steps)

print("Val steps:",val_steps)

Callbacks ready
Train steps: 5118
Val steps: 1095


In [ ]:
# ============================================================
# FIXED CELL 11 — Load Best Model + Beam Search Evaluation
# Fix: replace greedy decoding with beam search
#      this alone gives +0.05 to +0.10 BLEU-1 for free
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tqdm import tqdm


# ── Re-define custom layer before loading ──────────────────
class BahdanauAttention(tf.keras.layers.Layer):

    def __init__(self, units, **kwargs):
        super(BahdanauAttention, self).__init__(**kwargs)
        self.units = units
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V  = Dense(1)

    def call(self, features, hidden):
        hidden_expanded   = tf.expand_dims(hidden, 1)
        score             = tf.nn.tanh(self.W1(features) + self.W2(hidden_expanded))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)
        context_vector    = tf.reduce_sum(attention_weights * features, axis=1)
        return context_vector

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config


# Load best checkpoint
model = tf.keras.models.load_model(
    "/content/hybrid_best.keras",
    custom_objects={"BahdanauAttention": BahdanauAttention},
    compile=False
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)
print("Best model loaded ✅")


# ── Evaluate on test set ───────────────────────────────────
test_dataset_eval = tf.data.Dataset.from_generator(
    lambda: data_generator(test_images, BATCH_SIZE),
    output_signature=(
        (
            tf.TensorSpec(shape=(None, 64, 2048),  dtype=tf.float32),
            tf.TensorSpec(shape=(None, max_length), dtype=tf.float32)
        ),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)
    )
).prefetch(tf.data.AUTOTUNE)

test_results = model.evaluate(
    test_dataset_eval,
    steps=test_steps,
    verbose=1
)
print(f"\nTest Loss:     {test_results[0]:.4f}")
print(f"Test Accuracy: {test_results[1]:.4f}")


# ── Beam Search ────────────────────────────────────────────
def generate_caption_beam(model, feature, max_length, beam_width=5):
    """
    Beam search caption generation.
    Considers beam_width candidate sequences at every step
    instead of greedily picking the single best word.
    No-repeat-ngram prevents the repetition loops seen
    in greedy output ("aadmi aur mahila aur mahila...").
    """
    start_idx = tokenizer.word_index.get('startseq', 1)
    end_idx   = tokenizer.word_index.get('endseq',   2)

    # Each beam: (cumulative_neg_log_prob, [token_ids])
    beams     = [(0.0, [start_idx])]
    completed = []

    for _ in range(max_length):
        if not beams:
            break

        all_candidates = []

        for score, tokens in beams:
            # Stop expanding this beam if it hit endseq
            if tokens[-1] == end_idx:
                completed.append((score / len(tokens), tokens))
                continue

            seq  = pad_sequences([tokens], maxlen=max_length)
            feat = np.expand_dims(feature, 0)

            preds = model.predict([feat, seq], verbose=0)[0]  # (vocab_size,)

            # No-repeat trigram penalty — prevents "X Y X Y" loops
            if len(tokens) >= 3:
                for i in range(len(tokens) - 2):
                    if tokens[i] == tokens[-2] and tokens[i+1] == tokens[-1]:
                        # This trigram already seen — zero out likely continuations
                        next_word = tokens[i+2]
                        preds[next_word] *= 0.01

            # Expand top beam_width words
            top_indices = np.argsort(preds)[-beam_width:]

            for idx in top_indices:
                prob = float(preds[idx])
                if prob < 1e-10:
                    continue
                new_score = score - np.log(prob)
                all_candidates.append(
                    (new_score, tokens + [int(idx)])
                )

        # Keep best beam_width beams (lowest score = best)
        all_candidates.sort(key=lambda x: x[0])
        beams = all_candidates[:beam_width]

    # Add remaining active beams to completed
    completed += [(s / max(len(t), 1), t) for s, t in beams]

    if not completed:
        return ''

    # Pick sequence with lowest length-normalized score
    completed.sort(key=lambda x: x[0])
    best_tokens = completed[0][1]

    # Decode tokens to words
    words = []
    for idx in best_tokens:
        word = tokenizer.index_word.get(idx, '')
        if word == 'endseq':
            break
        if word not in ('startseq', ''):
            words.append(word)

    return ' '.join(words)


# ── BLEU Evaluation ────────────────────────────────────────
print("\nComputing BLEU scores (beam_width=5, 500 images)...")
print("This takes ~15 minutes — grab a coffee.\n")

smooth = SmoothingFunction().method1

actual    = []
predicted = []

sample_test = test_images[:500]

for img in tqdm(sample_test):
    if img not in features:
        continue

    feature = features[img]

    # All reference captions for this image
    refs = []
    for cap in captions[img]:
        words = [
            w for w in cap.split()
            if w not in ('startseq', 'endseq')
        ]
        if words:
            refs.append(words)

    if not refs:
        continue

    # Generated caption via beam search
    pred_caption = generate_caption_beam(
        model, feature, max_length, beam_width=5
    )
    pred_words = pred_caption.split()

    if pred_words:
        actual.append(refs)
        predicted.append(pred_words)


print("\n" + "=" * 45)
print("BLEU SCORES — Hybrid Model (beam search, width=5)")
print("=" * 45)

b1 = corpus_bleu(actual, predicted, weights=(1, 0, 0, 0),
                  smoothing_function=smooth)
b2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0),
                  smoothing_function=smooth)
b3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0),
                  smoothing_function=smooth)
b4 = corpus_bleu(actual, predicted,
                  smoothing_function=smooth)

print(f"BLEU-1: {b1:.4f}")
print(f"BLEU-2: {b2:.4f}")
print(f"BLEU-3: {b3:.4f}")
print(f"BLEU-4: {b4:.4f}")
print(f"Evaluated on: {len(predicted)} images")
print("=" * 45)

# Compare with old greedy scores
print("\nComparison:")
print(f"{'Metric':<10} {'Greedy (old)':>14} {'Beam=5 (new)':>14} {'Gain':>8}")
print("-" * 48)
# Replace the 0.XX below with your actual greedy scores
old_scores = {"BLEU-1": 0.43, "BLEU-2": 0.28, "BLEU-3": 0.18, "BLEU-4": 0.09}
new_scores = {"BLEU-1": b1,   "BLEU-2": b2,   "BLEU-3": b3,   "BLEU-4": b4}
for metric in ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]:
    old = old_scores[metric]
    new = new_scores[metric]
    print(f"{metric:<10} {old:>14.4f} {new:>14.4f} {new-old:>+8.4f}")


# ── Qualitative Examples ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import textwrap

print("\n\nQualitative examples (beam search vs reference):\n")

sample_imgs = random.sample(test_images[:200], min(6, len(test_images)))
fig, axes   = plt.subplots(2, 3, figsize=(18, 10))
axes        = axes.flatten()

for i, img_name in enumerate(sample_imgs):
    img_path = os.path.join(image_path, img_name)
    feature  = features[img_name]

    generated = generate_caption_beam(model, feature, max_length, beam_width=5)
    ref       = captions[img_name][0]
    ref       = ref.replace('startseq', '').replace('endseq', '').strip()

    ax = axes[i]
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')

    gen_text = '\n'.join(textwrap.wrap(f"Gen: {generated}", 42))
    ref_text = '\n'.join(textwrap.wrap(f"Ref: {ref}",       42))
    ax.set_title(f"{gen_text}\n\n{ref_text}", fontsize=8, loc='left', pad=6)

plt.suptitle(
    "Hybrid Model — Beam Search Captions vs References",
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig("/content/qualitative_beam.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: /content/qualitative_beam.png")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'bahdanau_attention_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Best model loaded ✅
1102/1102 ━━━━━━━━━━━━━━━━━━━━ 70s 62ms/step - accuracy: 0.2486 - loss: 5.9363

Test Loss:     5.9363
Test Accuracy: 0.2486

Computing BLEU scores (beam_width=5, 500 images)...
This takes ~15 minutes — grab a coffee.



 55%|█████▌    | 275/500 [58:33<37:47, 10.08s/it]

In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):

    def __init__(self, uni ts, **kwargs):
        super(BahdanauAttention, self).__init__(**kwargs)

        self.units = units

        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, features, hidden):

        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        score = tf.nn.tanh(
            self.W1(features) + self.W2(hidden_with_time_axis)
        )

        attention_weights = tf.nn.softmax(
            self.V(score),
            axis=1
        )

        context_vector = attention_weights * features

        context_vector = tf.reduce_sum(
            context_vector,
            axis=1
        )

        return context_vector

    def get_config(self):

        config = super().get_config()

        config.update({
            "units": self.units
        })

        return config

In [ ]:
model=tf.keras.models.load_model(

    "/content/hybrid_best.keras",

    custom_objects={
        "BahdanauAttention":BahdanauAttention
    },

    compile=False   # important
)

print("Best model loaded")

In [ ]:
model.compile(

    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.1
    ),

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    metrics=['accuracy']
)

In [ ]:
def generate_caption_greedy(model,feature,max_length):

    in_text='startseq'

    for i in range(max_length):

        seq=tokenizer.texts_to_sequences([in_text])[0]

        seq=pad_sequences([seq],maxlen=max_length)

        yhat=model.predict([feature,seq],verbose=0)

        yhat=np.argmax(yhat)

        word=tokenizer.index_word.get(yhat)

        if word is None:
            break

        in_text+=' '+word

        if word=='endseq':
            break

    return in_text

In [ ]:
img=test_images[25]

feature=features[img]

feature=np.expand_dims(feature,0)

print(generate_caption_greedy(
    model,
    feature,
    max_length
))

In [ ]:
from nltk.translate.bleu_score import corpus_bleu,SmoothingFunction
from tqdm import tqdm

smooth=SmoothingFunction().method1

actual=[]
predicted=[]

sample_test=test_images[:500]

for img in tqdm(sample_test):

    feature=features[img]

    feature=np.expand_dims(feature,0)


    refs=[]

    for cap in captions[img]:

        words=cap.split()

        # remove tokens
        words=words[1:-1]

        refs.append(words)


    actual.append(refs)


    pred=generate_caption_greedy(
        model,
        feature,
        max_length
    ).split()


    pred=pred[1:-1]

    predicted.append(pred)



print("\nBLEU SCORES")


print("BLEU-1:",
corpus_bleu(
actual,
predicted,
weights=(1,0,0,0),
smoothing_function=smooth
))


print("BLEU-2:",
corpus_bleu(
actual,
predicted,
weights=(0.5,0.5,0,0),
smoothing_function=smooth
))


print("BLEU-4:",
corpus_bleu(
actual,
predicted,
smoothing_function=smooth
))

In [ ]:
import shutil

# Save best model to Drive
shutil.copy(
    "/content/hybrid_best.keras",
    "/content/drive/MyDrive/hybrid_best.keras"
)

# Save features
shutil.copy(
    "/content/spatial_features.npy",
    "/content/drive/MyDrive/spatial_features.npy"
)

# Save tokenizer
shutil.copy(
    "/content/tokenizer.pkl",
    "/content/drive/MyDrive/tokenizer.pkl"
)

print("Everything saved to Drive")

In [ ]:
import shutil

# Save best model to Drive
shutil.copy(
    "/content/hybrid_best.keras",
    "/content/drive/MyDrive/hybrid_best.keras"
)

# Save features
shutil.copy(
    "/content/spatial_features.npy",
    "/content/drive/MyDrive/spatial_features.npy"
)

# Save tokenizer
shutil.copy(
    "/content/tokenizer.pkl",
    "/content/drive/MyDrive/tokenizer.pkl"
)

print("Everything saved to Drive")

In [ ]:
import pickle

with open("/content/drive/MyDrive/hybrid_history.pkl","wb") as f:
    pickle.dump(history.history,f)